# Install Dependencies and Import

In [ ]:
# Install dependencies
!pip install -q --upgrade numerapi pandas pyarrow matplotlib lightgbm scikit-learn scipy cloudpickle optuna
!pip install -q --no-deps numerai-tools

!pip install -q --upgrade numerapi numerai-tools optuna xgboost catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import timedelta
import time

from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

import json
import os
import gc
import shutil
import itertools
from tqdm import tqdm
import random

import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer
import cloudpickle
import pickle
import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:.3f}'.format
pd.options.display.max_columns = 500

# Inline plots
%matplotlib inline

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Functions

In [ ]:
# Общая информация о датафрейме
def describe_dataframe(df: pd.DataFrame):

    n_rows, n_cols = df.shape
    missing_total = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()

    print(f"Размер: {n_rows} × {n_cols}")
    print(f"Пропуски: {missing_total}")
    print(f"Дубликаты: {duplicates}")

# Информация по столбцам
def describe_columns(df: pd.DataFrame) -> pd.DataFrame:

    desc = pd.DataFrame({
        "dtype": df.dtypes,
        "min": df.min(numeric_only=True),
        "max": df.max(numeric_only=True),
        "missing": df.isnull().sum(),
    })
    desc["missing_%"] = desc["missing"] / len(df) * 100

    return desc

In [ ]:
def save_experiment(model, model_name, validation_data, results_df):
    """
    Сохраняет модель, validation с предсказаниями и таблицу результатов в файлы.

    Параметры:
        model: обученная модель (например, LGBMRegressor, XGBRegressor)
        validation_data: pd.DataFrame с target и предсказаниями
        results_df: pd.DataFrame с метриками
        prefix: префикс для имён файлов (по умолчанию "experiment")
    """
    # Имена файлов
    model_file = f"/content/drive/MyDrive/Colab Notebooks/{model_name}.pkl"
    validation_file = "/content/drive/MyDrive/Colab Notebooks/val_med_pred.pkl"
    results_file = "/content/drive/MyDrive/Colab Notebooks/results_med.pkl"

    # Сохранение
    with open(model_file, "wb") as f:
        pickle.dump(model, f)
    # print(f"Модель сохранена в {model_file}")

    with open(validation_file, "wb") as f:
        pickle.dump(validation_data, f)
    # print(f"Validation с предсказаниями сохранён в {validation_file}")

    with open(results_file, "wb") as f:
        pickle.dump(results_df, f)
    # print(f"Таблица результатов сохранена в {results_file}")

save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)



# Datasets

In [ ]:
# Load Train Dataset
train = pd.read_parquet(f"{DATA_VERSION}/train.parquet", columns=["era", "target"] + feature_set)
train['era'] = train['era'].astype(int)

describe_dataframe(train)
describe_columns(train).head(3)

train_benchmark_models = pd.read_parquet(f"{DATA_VERSION}/train_benchmark_models.parquet")
train_benchmark_models['era'] = train_benchmark_models['era'].astype(int)

# Join Train & train_benchmark_models
train = train.join(train_benchmark_models[["v5_lgbm_cyrusd20",	"v5_lgbm_teager2b20",	"v5_lgbm_ct_blend"]], how="left")

# Сколько строк в каждом датафрейме
print(f"Строк в train: {len(train)}")
print(f"Строк в train_benchmark_models: {len(train_benchmark_models)}")

# Совпадающие id (индексы)
common_ids = train.index.intersection(train_benchmark_models.index)
print(f"Совпадающих id: {len(common_ids)}")

In [ ]:
# Load Validation Dataset
validation = pd.read_parquet(f"{DATA_VERSION}/validation.parquet", columns=["era", "data_type", "target"] + feature_set)
validation['era'] = validation['era'].astype(int)

describe_dataframe(validation)
describe_columns(validation).head(3)

validation = validation[validation["data_type"] == "validation"]
del validation["data_type"]

validation_benchmark_models = pd.read_parquet(f"{DATA_VERSION}/validation_benchmark_models.parquet")
validation_benchmark_models['era'] = validation_benchmark_models['era'].astype(int)

# Join Validation & validation_benchmark_models
validation = validation.join(validation_benchmark_models[["v5_lgbm_cyrusd20",	"v5_lgbm_teager2b20",	"v5_lgbm_ct_blend"]], how="left")

# Сколько строк в каждом датафрейме
print(f"Строк в train: {len(validation)}")
print(f"Строк в train_benchmark_models: {len(validation_benchmark_models)}")

# Совпадающие id (индексы)
common_ids = validation.index.intersection(validation_benchmark_models.index)
print(f"Совпадающих id: {len(common_ids)}")

# Embargo the first 4 eras
last_era = int(train["era"].unique()[-1])
eras_embargo = [era for era in [last_era + i for i in range(1, 5)]]
validation = validation[~validation4["era"].isin(eras_embargo)]

print(f'Train eras: {train.era.min()}-{train.era.max()} ({train["era"].nunique()})')
print(f'Validation eras: {validation.era.min()}-{validation.era.max()} ({validation["era"].nunique()})')
print(f'Eras embrago: {eras_embargo}')

# Meta Model
meta_model = pd.read_parquet("v4.3/meta_model.parquet")
meta_model['era'] = meta_model['era'].astype(int)

describe_dataframe(meta_model)
describe_columns(meta_model)

# Join numerai_meta_model and validation
validation = validation.join(meta_model["numerai_meta_model"], how="left")
validation.rename(columns={"numerai_meta_model": "meta_model"}, inplace=True)

# Сколько строк в каждом датафрейме
print(f"Строк в validation: {len(validation)}")
print(f"Строк в meta_model: {len(meta_model)}")

# Совпадающие id (индексы)
common_ids = validation.index.intersection(meta_model.index)
print(f"Совпадающих id: {len(common_ids)}")

In [ ]:
# Downsample to every 4th era
train4 = train[train["era"].isin(train["era"].unique()[::4])]
validation4 = validation[validation["era"].isin(validation["era"].unique()[::4])]

In [ ]:
# Распределение по эрам (строк на эру)
train.groupby("era").size().plot(
    title="Number of rows per era",
    figsize=(5, 3),
    xlabel="Era")

validation.groupby("era").size().plot(
    title="Number of rows per era",
    figsize=(5, 3),
    xlabel="Era")

# Metrics

In [ ]:
def per_era_merics(df, era="era", target="target", prediction="prediction"):

    per_era_corr = df.groupby(era).apply(lambda x: numerai_corr(x[[prediction]], x[target]))
    per_era_mmc = df.dropna().groupby(era).apply(lambda x: correlation_contribution(x[[prediction]], x['meta_model'], x[target]))

    corr_mean = round(per_era_corr[prediction].mean(), 3)
    corr_std = round(per_era_corr[prediction].std(ddof=0), 3)
    corr_sharpe = round(corr_mean / corr_std if corr_std != 0 else np.nan, 3)
    corr_max_drawdown = round(((per_era_corr[prediction].cumsum().expanding().max() - per_era_corr[prediction].cumsum()).max()), 3)

    mmc_mean = round(per_era_mmc[prediction].mean(), 3)
    mmc_std = round(per_era_mmc[prediction].std(ddof=0), 3)
    mmc_sharpe = round(mmc_mean / mmc_std, 3)
    mmc_max_drawdown = round((per_era_mmc[prediction].cumsum().expanding(min_periods=1).max() - per_era_mmc[prediction].cumsum()).max(), 3)

    metrics = {
        "corr_mean": float(corr_mean),
        "corr_std": corr_std,
        "corr_sharpe": float(corr_sharpe),
        "corr_max_drawdown": corr_max_drawdown,
        "mmc_mean": float(mmc_mean),
        "mmc_std": mmc_std,
        "mmc_sharpe": float(mmc_sharpe),
        "mmc_max_drawdown": mmc_max_drawdown
    }

    return metrics, per_era_corr, per_era_mmc


metrics, per_era_corr, per_era_mmc = per_era_merics(validation)
metrics_df = pd.DataFrame({'model': metrics})

In [ ]:
def plot_per_era_metrics(per_era_corr_df, per_era_mmc_df):

    fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=False)

    # Plotting function for bar and cumulative plots
    def plot_metrics(ax_bar, ax_cum, data, title, color, label):
        ax_bar.bar(range(len(data)), data.values, color=color, alpha=0.7, label=label)
        ax_bar.set_title(title)
        ax_bar.set_xlabel("Era")
        ax_bar.set_ylabel(label.split()[-1])
        ax_bar.grid(alpha=0.3)

        ax_cum.plot(range(len(data)), data.cumsum().values, color=color, linewidth=2, label="Cumulative " + label.split()[-1])
        ax_cum.set_title("Cumulative " + label.split()[-1])
        ax_cum.set_xlabel("Era")
        ax_cum.set_ylabel(label.split()[-1])
        ax_cum.grid(alpha=0.3)

    # Plotting CORR
    plot_metrics(axes[0, 0], axes[1, 0], per_era_corr_df['prediction'], "Validation CORR", 'darkorange', "Validation CORR")

    # Plotting MMC
    plot_metrics(axes[0, 1], axes[1, 1], per_era_mmc_df['prediction'], "Validation MMC", 'purple', "Validation MMC")

    plt.show()

In [ ]:
def plot_per_era_metrics(per_era_corr_df, per_era_mmc_df=None):

    fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=False)

    ax1 = axes[0, 0]
    ax1.bar(range(len(per_era_corr_df)), per_era_corr_df['prediction'].values, color='darkorange', alpha=0.7, label="Validation CORR")
    ax1.set_title("Validation CORR")
    ax1.set_xlabel("Era")
    ax1.set_ylabel("CORR")
    ax1.grid(alpha=0.3)

    if per_era_mmc_df:
      ax2 = axes[0, 1]
      ax2.bar(range(len(per_era_mmc_df)), per_era_mmc_df['predict'].values, color='purple', alpha=0.7, label="Validation MMC")
      ax2.set_title("Validation MMC")
      ax2.set_xlabel("Era")
      ax2.set_ylabel("MMC")
      ax2.grid(alpha=0.3)

    ax3 = axes[1, 0]
    ax3.plot(range(len(per_era_corr_df)), per_era_corr_df['prediction'].cumsum().values, color="darkorange", linewidth=2, label="Cumulative CORR")
    ax3.set_title("Cumulative CORR")
    ax3.set_xlabel("Era")
    ax3.set_ylabel("CORR")
    ax3.grid(alpha=0.3)

    if per_era_mmc_df:
      ax4 = axes[1, 1]
      ax4.plot(range(len(per_era_mmc_df)), per_era_mmc_df['predict'].cumsum().values, color="purple", linewidth=2, label="Cumulative MMC")
      ax4.set_title("Cumulative MMC")
      ax4.set_xlabel("Era")
      ax4.set_ylabel("MMC")
      ax4.grid(alpha=0.3)

    plt.show()

In [ ]:
# Plot the per-era correlation

fig, axes = plt.subplots(2, 2, figsize=(18,10), sharex=False)

ax1 = axes[0,0]
ax1.bar(range(len(per_era_corr)), per_era_corr['predict'].values, color='darkorange', alpha=0.7, label="Validation CORR")
ax1.set_title("Validation CORR")
ax1.set_xlabel("Era")
ax1.set_ylabel("CORR")
ax1.grid(alpha=0.3)

ax2 = axes[0,1]
ax2.bar(range(len(per_era_mmc)), per_era_mmc['predict'].values, color='purple', alpha=0.7, label="Validation MMC")
ax2.set_title("Validation MMC")
ax2.set_xlabel("Era")
ax2.set_ylabel("MMC")
ax2.grid(alpha=0.3)

ax3 = axes[1,0]
ax3.plot(range(len(per_era_corr)), per_era_corr['predict'].cumsum().values, color="darkorange", linewidth=2, label="Cumulative CORR")
ax3.set_title("Cumulative CORR")
ax3.set_xlabel("Era")
ax3.set_ylabel("CORR")
ax3.grid(alpha=0.3)

ax4 = axes[1,1]
ax4.plot(range(len(per_era_mmc)), per_era_mmc['predict'].cumsum().values, color="purple", linewidth=2, label="Cumulative MMC")
ax4.set_title("Cumulative MMC")
ax4.set_xlabel("Era")
ax4.set_ylabel("MMC")
ax4.grid(alpha=0.3)

plt.show()

# Training

In [ ]:
# Универсальная функция обучения
def train_and_evaluate(model_name, model_class, params, train_df, val_df, features, target="target"):
    print(f"\n🔹 Обучаем модель: {model_name}")
    model = model_class(**params)
    model.fit(train_df[features], train_df[target])
    val_df[model_name] = model.predict(val_df[features])
    
    metrics, per_era_corr, per_era_mmc = per_era_metrics(val_df, model_name)
    log_result(model_name, metrics, params)
    print(f"Результаты {model_name}: corr_sharpe={metrics['corr_sharpe']}, mmc_sharpe={metrics['mmc_sharpe']}")
    gc.collect()
    return model, metrics


# Обучение всех моделей
for cfg in model_configs:
    model, metrics = train_and_evaluate(
        model_name=cfg["name"],
        model_class=cfg["class"],
        params=cfg["params"],
        train_df=train,
        val_df=validation,
        features=feature_set
    )

In [ ]:
def era_wise_cv(train_df, feature_set, params, model_type, embargo=4, n_splits=5):
    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=embargo)

    results = []
    all_gain_importances = []
    all_split_importances = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(eras), 1):
        train_eras = [eras[i] for i in train_idx]
        val_eras = [eras[i] for i in val_idx]

        print(f"\nFold {fold}: Train {train_eras[0]}–{train_eras[-1]}, Val {val_eras[0]}–{val_eras[-1]}")

        train_split = train_df[train_df["era"].isin(train_eras)].copy()
        val_split  = train_df[train_df["era"].isin(val_eras)].copy()
        
        X_train, y_train = train_split[feature_set], train_split["target"]
        X_val, y_val     = val_split[feature_set], val_split["target"]

        # --- обучение модели ---
        model = model_type(**params, random_state=SEED)
        model.fit(X_train, y_train)

        val_split["prediction"] = model.predict(X_val)

        # --- метрики per-era ---
        per_era_corr = val_split.groupby("era").apply(
            lambda x: numerai_corr(x[["prediction"]], x["target"])
        )["prediction"]

        corr_mean = per_era_corr.mean()
        corr_std  = per_era_corr.std(ddof=0)
        corr_sharpe = corr_mean / corr_std if corr_std != 0 else np.nan

        results.append({
            "fold": str(fold),
            "train_eras": f"{train_eras[0]}-{train_eras[-1]}",
            "val_eras":   f"{val_eras[0]}-{val_eras[-1]}",
            "corr_sharpe": round(corr_sharpe, 3),
            "corr_mean":   round(corr_mean, 3),
            "corr_std":    round(corr_std, 3),
        })

        # --- importance через универсальную функцию ---
        fi_df = get_lgbm_feature_importance(model)
        gain_df  = fi_df.set_index("feature")["gain"]
        split_df = fi_df.set_index("feature")["split"]

        all_gain_importances.append(gain_df)
        all_split_importances.append(split_df)


        del model
        gc.collect()

    # Сборка результатов CV
    cv_results = pd.DataFrame(results)
    cv_mean = cv_results.mean(numeric_only=True).round(3).to_dict()

    # --- усреднение importance по фолдам ---
    gain_df  = pd.concat(all_gain_importances, axis=1).mean(axis=1).rename("gain")
    split_df = pd.concat(all_split_importances, axis=1).mean(axis=1).rename("split")

    # Сборка важности признаков
    importance_df = pd.concat([gain_df, split_df], axis=1)
    importance_df = importance_df.sort_values("gain", ascending=False).reset_index()

    return cv_results, cv_mean, importance_df


In [ ]:
import xgboost as xgb
import time
from datetime import timedelta
import pandas as pd

# Начнём отсчёт времени
start_time = time.time()

# Параметры XGBoost (аналогичные параметрам LightGBM)
params = {
    'objective': 'reg:squarederror',       # Цель регрессии (квадратичная ошибка)
    'eval_metric': ['rmse'],               # Метрика оценки RMSE
    'booster': 'gbtree',                   # Тип бустера (дерево)
    'n_estimators': 20000,                 # Количество деревьев
    'eta': 0.001,                          # Скорость обучения (эквивалент learning_rate)
    'max_depth': 6,                        # Максимальная глубина дерева
    'subsample': 0.1,                      # Аналог colsample_bytree
    'gamma': 0,                            # Минимальное снижение потерь для разделения узла
    'lambda': 1,                           # L2 регуляризация
    'alpha': 0,                            # L1 регуляризация
    'tree_method': 'gpu_hist',              # Использование GPU
    'predictor': 'gpu_predictor',           # Предиктор для предсказания на GPU
    'random_state': 42                     # Случайное начальное состояние
}

# Создаем экземпляр модели XGBoost
xgboost_regressor = xgb.XGBRegressor(**params)

# Подгонка модели на обучающих данных
xgboost_regressor.fit(train[feature_set], train['target'])

# Прогнозирование на валидационном наборе
validation['prediction'] = xgboost_regressor.predict(validation[feature_set])

# Расчет метрик
metrics_basic, per_era_corr, per_era_mmc = per_era_merics(validation)

# Время выполнения
elapsed_time = time.time() - start_time
formatted_time = str(timedelta(seconds=int(elapsed_time)))
print(f"Время выполнения: {formatted_time}")

# Преобразование результатов в dataframe
metrics_basic = pd.DataFrame({'basic_standart_small': metrics_basic})
main_metrics = pd.concat([main_metrics, metrics_basic], axis=1)
main_metrics.T

# GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search over
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2'],
    'bootstrap': [True, False]
}

grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=5)
grid_search.fit(X_train, y_train)

In [ ]:
# --- Кастомный скорер ---
def numerai_corr_scorer(y_true, y_pred):
    df = pd.DataFrame({"prediction": y_pred, "target": y_true})
    return float(numerai_corr(df["prediction"], df["target"]))

numerai_scorer = make_scorer(numerai_corr_scorer, greater_is_better=True)


# --- Инициализация модели ---
model = lgb.LGBMRegressor(objective="regression", n_jobs=-1, device_type="gpu")

param_grid = {
    "learning_rate": [0.01, 0.02],
    "num_leaves": [15, 31],
    "max_depth": [3, 5],
    "colsample_bytree": [0.1, 0.3],
}

# --- TimeSeriesSplit ---
tscv = TimeSeriesSplit(n_splits=5)

# --- GridSearchCV ---
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=numerai_scorer,
    cv=tscv,
    verbose=2,
    n_jobs=-1,
)

# --- Обучение ---
X = train[feature_set]
y = train["target"]
# X = train[feature_set].to_numpy()
# y = train["target"].to_numpy()

grid.fit(X, y)

# --- Результаты ---
print("\n✅ Лучшие параметры:")
print(grid.best_params_)

print("\n🏆 Лучший скор (numerai_corr):")
print(grid.best_score_)

# Полная таблица результатов
results_df = pd.DataFrame(grid.cv_results_)
results_df = results_df.sort_values(by="mean_test_score", ascending=False)
results_df.head(10)

In [ ]:
def grid_search_era_wise(train_df, feature_set, param_grid, n_splits, embargo):

    """Перебор параметров по Sharpe Ratio."""
    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    all_results = []
    best_score = -np.inf
    best_params = None

    for i, params_subset in enumerate(param_combinations, 1):
        params = {
            #"n_estimators": 10,
            #"learning_rate": 0.1,
            #"max_depth": 3,
            #"num_leaves": 5,
            "colsample_bytree": 0.1,
            "verbosity": -1,
            "device_type": "gpu",
            **params_subset,
        }

        print(f"\n🔎 [{i}/{len(param_combinations)}] Проверяем: {params_subset}")
        results_df, _ = era_wise_cv_with_importance(train_df, feature_set, n_splits=n_splits, params=params, embargo=embargo)

        mean_sharpe = results_df["corr_sharpe"].iloc[:-1].mean()
        mean_corr = results_df["corr_mean"].iloc[:-1].mean()

        all_results.append({
            **params_subset,
            "mean_corr": mean_corr,
            "mean_sharpe": mean_sharpe,
        })

        if mean_sharpe > best_score:
            best_score = mean_sharpe
            best_params = params_subset

        gc.collect()

    search_results = pd.DataFrame(all_results).sort_values(by="mean_sharpe", ascending=False).reset_index(drop=True)

    print(f"\n🏆 Лучшие параметры: {best_params} с Sharpe = {best_score:.4f}")
    return best_params, search_results


param_grid = {
    "n_estimators": [2000, 20000],
    "max_depth": [5, 6],
    "learning_rate": [0.01, 0.001],
    "num_leaves": [31, 64]
    }

best_params, search_results = grid_search_era_wise(train, feature_set, param_grid, n_splits=2, embargo=4)
search_results

# Feature Importance

In [ ]:
# Среднее, стандартное отклонение и коэффициент вариации
importance_stats = importances_df.groupby("feature")["importance"].agg(["mean", "std"])
importance_stats["cv"] = importance_stats["std"] / (importance_stats["mean"] + 1e-6)

# Сортировка по средней важности
importance_stats = importance_stats.sort_values(by="mean", ascending=False)

display(importance_stats)

top_imp = importance_stats.sort_values(by="mean", ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(top_imp.index, top_imp["mean"], xerr=top_imp["std"])
plt.gca().invert_yaxis()
plt.title("Feature Importance Stability (mean ± std)")
plt.xlabel("Mean Importance")
plt.show()

In [ ]:
# Пороговые значения — их можно подбирать
MIN_MEAN_IMPORTANCE = 1.0        # минимальная средняя важность признака
MAX_CV_IMPORTANCE = 0.5          # максимальный коэффициент вариации (нестабильность)

# Добавим фильтр
filtered_features = (
    importance_stats[
        (importance_stats["mean"] > MIN_MEAN_IMPORTANCE) &
        (importance_stats["cv"] < MAX_CV_IMPORTANCE)
    ]
    .sort_values(by="mean", ascending=False)
    .index.tolist()
)

print(f"✅ Выбрано стабильных признаков: {len(filtered_features)} из {len(feature_set)}")

filtered_features

In [ ]:
importance = pd.DataFrame({
    'feature': feature_set,
    'importance': lgbm.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.gca().invert_yaxis()
plt.title('Важность признаков (Feature importance)')
plt.xlabel('Вклад в модель')
plt.show()

# LGBM
lgb.plot_importance(lgbm, importance_type='gain')
lgb.plot_importance(lgbm, importance_type='split')

# Feature Neutralization

In [ ]:
# Compute the per-era correlation of each serenity feature to the target
per_era_corr = train.groupby("era").apply(
    lambda d: numerai_corr(d[feature_set], d["target"])
)

# Flip sign for negative mean correlation since we only care about magnitude
per_era_corr *= np.sign(per_era_corr.mean())

# Plot the per-era correlations
per_era_corr.cumsum().plot(
    title="Cumulative Absolute Value CORR of Features and the Target",
    figsize=(15, 5),
    legend=False,
    xlabel="Era"
  )

plt.show()

In [ ]:
def get_features_metrics(corr):
    corr_mean = corr.mean()
    corr_std = corr.std(ddof=0)
    corr_sharpe = corr_mean / corr_std
    max_drawdown = -(corr.cumsum().expanding(min_periods=1).max() - corr.cumsum()).max()

    eras = train.era.unique()
    halfway_era = len(eras)//2
    corr_mean_first_half = corr.loc[eras[:halfway_era]].mean()
    corr_mean_second_half = corr.loc[eras[halfway_era:]].mean()
    delta = abs(corr_mean_first_half - corr_mean_second_half)

    return {
      "mean": corr_mean,
      "std": corr_std,
      "sharpe": corr_sharpe,
      "max_drawdown": max_drawdown,
      "delta": delta
    }

# compute performance metrics for each feature
feature_metrics = [
    get_features_metrics(per_era_corr[feature_name])
    for feature_name in feature_set
]

# convert to numeric DataFrame and sort
feature_metrics = (
    pd.DataFrame(feature_metrics, index=feature_set)
    .apply(pd.to_numeric)
    .sort_values("mean", ascending=False)
)

# plot the performance metrics of the features as bar charts sorted by mean
feature_metrics.sort_values('mean', ascending=False).plot.bar(
    title="Performance Metrics of Features Sorted by Mean",
    subplots=True,
    figsize=(15, 6),
    layout=(2, 3),
    sharex=False,
    xticks=[],
    snap=False
)

In [ ]:
# plot the per era correlation of the feature with the highest vs lowest std
per_era_corr[[feature_metrics["std"].idxmin(), feature_metrics["std"].idxmax()]].plot(
    figsize=(15, 5), title="Per-era Correlation of Features to the Target", xlabel="Era"
)
plt.legend(["lowest std", "highest std"])


# plot the cumulative per era correlation of the feature with the highest vs lowest delta
per_era_corr[[feature_metrics["delta"].idxmin(), feature_metrics["delta"].idxmax()]].cumsum().plot(
    figsize=(15, 5), title="Cumulative Correlation of Features to the Target", xlabel="Era"
)
plt.legend(["lowest delta", "highest delta"])


# plot the cumulative per era correlation of the feature with the highest vs lowest max_drawdown
per_era_corr[[feature_metrics["max_drawdown"].idxmax(), feature_metrics["max_drawdown"].idxmin()]].cumsum().plot(
    figsize=(15, 5), title="Cumulative Correlation of Features to the Target", xlabel="Era"
)
plt.legend(["lowest max_drawdown", "highest max_drawdown"])



# Создаем фигуру с четырьмя подплотами
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plt.suptitle("Cumulative CORR of Features to the Target", fontsize=16)

# Std
axes[0, 0].plot(per_era_corr[[feature_metrics["std"].idxmin(), feature_metrics["std"].idxmax()]])  # Используем plot() для линий
axes[0, 0].legend(["lowest std", "highest std"], loc="upper left")
axes[0, 0].set_title("Std")
axes[0, 0].set_xlabel("Era")
axes[0, 0].grid(True, alpha=0.3)

# Sharpe
axes[0, 1].plot(per_era_corr[[feature_metrics["sharpe"].idxmax(), feature_metrics["sharpe"].idxmin()]].cumsum())
axes[0, 1].legend(["lowest max_sharpe", "highest max_sharpe"], loc="upper left")
axes[0, 1].set_title("Sharpe")
axes[0, 1].set_xlabel("Era")
axes[0, 1].grid(True, alpha=0.3)

# Max Drawdown
axes[1, 0].plot(per_era_corr[[feature_metrics["max_drawdown"].idxmax(), feature_metrics["max_drawdown"].idxmin()]].cumsum())
axes[1, 0].legend(["lowest max_drawdown", "highest max_drawdown"], loc="upper left")
axes[1, 0].set_title("Max Drawdown")
axes[1, 0].set_xlabel("Era")
axes[1, 0].grid(True, alpha=0.3)

# Delta
axes[1, 1].plot(per_era_corr[[feature_metrics["delta"].idxmin(), feature_metrics["delta"].idxmax()]].cumsum())
axes[1, 1].legend(["lowest delta", "highest delta"], loc="upper left")
axes[1, 1].set_title("Delta")
axes[1, 1].set_xlabel("Era")
axes[1, 1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

In [ ]:
# Compute the Pearson correlation of the predictions with each of the serenity features of the small feature set
feature_exposures = validation.groupby("era").apply(
    lambda d: d[feature_set].corrwith(d["predict"])
)

# Plot the feature exposures as bar charts
feature_exposures.plot.bar(
    title="Feature Exposures",
    figsize=(16, 10),
    layout=(10, 5), 
    xticks=[],
    subplots=True,
    sharex=False,
    legend=False,
    snap=False
)

for ax in plt.gcf().axes:
    ax.set_xlabel("")
    ax.title.set_fontsize(10)

plt.tight_layout(pad=1.5)
plt.gcf().suptitle("Feature Exposures", fontsize=15)
plt.show()

In [ ]:
# Plot the max feature exposure per era
max_feature_exposure = feature_exposures.max(axis=1)
max_feature_exposure.plot(
  title="Max Feature Exposure",
  kind="bar",
  figsize=(10, 5),
  xticks=[],
  snap=False
)
# Mean max feature exposure across eras
print("Mean of max feature exposure", max_feature_exposure.mean())

In [ ]:
# import neutralization from numerai-tools
from numerai_tools.scoring import neutralize

# Neutralize predictions per-era against features at different proportions
proportions = [0.25, 0.5, 0.75, 1.0]
for proportion in proportions:
    neutralized = validation.groupby("era", group_keys=True).apply(
        lambda d: neutralize(
          d[["predict"]],
          d[feature_set],
          proportion=proportion
        )
    ).reset_index().set_index("id")
    validation[f"neutralized_{proportion*100:.0f}"] = neutralized["predict"]

# Align the neutralized predictions with the validation data
prediction_cols = ["predict"] + [f for f in validation.columns if "neutralized" in f]
validation[["era", "target"] + prediction_cols]

In [ ]:
# Compute max feature exposure for each set of predictions
max_feature_exposures = pd.concat([
    validation.groupby("era").apply(
        lambda d: d[feature_set].corrwith(d[col]).abs().max()
    ).rename(col)
    for col in prediction_cols
], axis=1)

# print mean feature exposure of each proportion
print('mean feature exposures:')
print(round(max_feature_exposures.mean(), 3))

# Plot max feature exposures
max_feature_exposures.plot.bar(
  title="Max Feature Exposures",
  figsize=(10, 5),
  xticks=[],
  snap=False
)

In [ ]:
# calculate per-era CORR for each set of predictions
correlations = validation.groupby("era").apply(
    lambda d: numerai_corr(d[prediction_cols], d["predict"])
)

# calculate the cumulative corr across eras for each neutralization proportion
cumulative_correlations = correlations.cumsum().sort_index()

# Show the cumulative correlations
pd.DataFrame(cumulative_correlations).plot(
    title="Cumulative Correlation of Neutralized Predictions",
    figsize=(10, 6),
    xticks=[]
)

In [ ]:
summary_metrics = {}
for col in prediction_cols:
    mean = correlations[col].mean()
    std = correlations[col].std(ddof=0)
    sharpe = mean / std
    rolling_max = cumulative_correlations[col].expanding(min_periods=1).max()
    max_drawdown = (rolling_max - cumulative_correlations[col]).max()
    summary_metrics[col] = {
        "mean": mean,
        "std": std,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
    }

pd.DataFrame(summary_metrics).T

# Fast Neutralization

In [ ]:
# Создаём колонку для нейтрализации
validation["meta_model_clean"] = validation["meta_model"].fillna(validation["meta_model"].mean())

# Нейтрализация по meta_model
proportions = [0.25, 0.5, 0.75, 1.0]
for proportion in proportions:
    pred_col = model_name
    neut_col = f"{model_name}_neut{proportion*100:.0f}"
    
    validation[neut_col] = neutralize(
        validation[[pred_col]],         
        validation[["meta_model_clean"]],   
        proportion=proportion           
    )[pred_col]

    # Расчёт метрик для нейтрализованной версии
    metrics_neut = per_era_metrics(validation, pred_col=neut_col)

    new_row = {"model_name": neut_col, **metrics_neut}
    neut_df = pd.concat([neut_df, pd.DataFrame([new_row])], ignore_index=True)


row = results_df.loc[results_df["model_name"] == "baseline"]
neut_df = pd.concat([row, neut_df], ignore_index=True)
neut_df

# Linear Regression

In [ ]:
# Linear Regression
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()

cross_validate

cv_lr = cross_validate(model_lr, train[feature_set], train['target'], scoring=rmsle, cv=5, return_train_score=True)

# Преобразование MSE в RMSLE
rmsle_test = np.sqrt(-cv_lr['test_score'])
rmsle_train = np.sqrt(-cv_lr['train_score'])

result_lr = np.mean(rmsle_test)

print(f'Средний RMSLE на TrainFold: {np.mean(rmsle_train):.4f}')
print(f'Средний RMSLE на TestFold: {np.mean(rmsle_test):.4f}')


# --- Кастомный скорер ---
def numerai_corr_scorer(y_true, y_pred):
    df = pd.DataFrame({"prediction": y_pred, "target": y_true})
    return float(numerai_corr(df["prediction"], df["target"]))

numerai_scorer = make_scorer(numerai_corr_scorer, greater_is_better=True)
numerai_scorer

model_lr = LinearRegression()

tscv = TimeSeriesSplit(n_splits=2)
cv_lr = cross_validate(model_lr, train[feature_set], train['target'], scoring=numerai_scorer,  cv=tscv, return_train_score=True)


X = train[feature_set]
y = train["target"]

model_lr.fit(X, y)

# Optuna

In [ ]:
def objective(trial):

    params_optuna = {
        # --- Основные параметры ---
        'n_estimators': trial.suggest_int('n_estimators', 8000, 12000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.02, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 7),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63, step=8),
        
        # --- Регуляризация ---
        'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 20.0, log=True),  
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),  
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.1),
        'min_child_samples': trial.suggest_int('min_child_samples', 100, 400, step=50),
        
        # --- Сэмплирование ---
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.03, 0.12),

        'verbosity': -1,
        'device_type': 'gpu',
    }

    # Запускаем кросс-валидацию
    cv_result = era_wise_cv_optuna(
        train_df=train,
        feature_set=feature_set,
        params=params_optuna,
        n_splits=3,
        esr=100 
    )

    trial.set_user_attr("best_iteration", cv_result["best_iteration"])

    return cv_result["corr_sharpe"]

In [ ]:
# --- ЗАПУСК ПОДБОРА ГИПЕРПАРАМЕТРОВ --- 
# 210 min

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="maximize", study_name="lgbm-tuning")

# Запускаем оптимизацию
study.optimize(objective, n_trials=50)

results = study.trials_dataframe()
results = results.sort_values("value", ascending=False)

print("Best Params:", study.best_params)
print("Best Corr Sharpe:", study.best_value)
print("Best Iteration:", study.best_trial.user_attrs["best_iteration"])

display(results[["number", "value", "state"] + [c for c in results.columns if "params_" in c]].head(10))

In [ ]:
print(f"Tuned Model Corr Sharpe: {results_df.loc[results_df['model_name'] == 'tuned', 'corr_sharpe'].values[0]}")
print(f"Optuna Best Corr Sharpe: {study.best_trial.value}")

plot_optimization_history(study)
plot_param_importances(study)
plot_contour(study, params=['learning_rate', 'max_depth']) 
plt.show()

In [ ]:
# LGBM Regressor 
model_name = "best_optuna"

best_params = study.best_params.copy()
best_params.update({
    'n_estimators': study.best_trial.user_attrs.get('best_iteration', 500),
    'verbose': -1
})


elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=best_params)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=best_params)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

In [ ]:
baseline_score = result['cv_summary']['corr_mean']

def objective(trial: Trial):
    
    # Tree structure: leaves bounded by depth
    max_depth = trial.suggest_int('max_depth', 4, 5)
    max_leaves = 2 ** max_depth
    num_leaves = trial.suggest_int('num_leaves', 2, max_leaves)

    params_optuna = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 300, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': num_leaves,
        'max_depth': max_depth,
        'colsample_bytree': 0.1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.7),
        # 'min_child_samples' : trial.suggest_int('min_child_samples', 20, 100)
    }

    score = era_wise_cv_optuna(params=params_optuna, train_df=train, trial=trial)

    return score

# Other

In [ ]:
class PurgedTimeSeriesSplit:
    """
    Временное разбиение с purge и embargo.
    Гарантирует ровно n_folds фолдов.
    """
    def __init__(self, n_folds=5, purge_gap=1, embargo=1):
        self.n_folds = n_folds
        self.purge_gap = purge_gap
        self.embargo = embargo

    def split(self, X, y=None, groups=None):
        # Уникальные эры в хронологическом порядке
        unique_eras = np.unique(groups)
        era_to_order = {era: i for i, era in enumerate(unique_eras)}
        group_order = np.array([era_to_order[era] for era in groups])

        n_eras = len(unique_eras)

        # Делим на (n_folds + 1) блоков: чтобы было n_folds валидаций
        split_indices = np.array_split(np.arange(n_eras), self.n_folds + 1)

        for i in range(1, self.n_folds + 1):
            if i >= len(split_indices):
                break

            # Обучающие эры: все блоки до i
            train_era_indices = np.concatenate(split_indices[:i])
            train_eras = unique_eras[train_era_indices]

            # Валидационные эры: i-й блок
            val_era_indices = split_indices[i]
            val_eras = unique_eras[val_era_indices]

            # Purge: удаляем последние `purge_gap` эр из train
            train_max_order = max(era_to_order[era] for era in train_eras)
            purged_max_order = train_max_order - self.purge_gap

            # Embargo: пропускаем первые `embargo` эр после purged_max_order
            min_val_order = purged_max_order + 1 + self.embargo

            # Оставляем только те val-эры, которые после embargo
            allowed_val_eras = [era for era in val_eras if era_to_order[era] >= min_val_order]

            # пропускаем, если валидация пуста
            if len(allowed_val_eras) == 0:
                continue  

            # Маски по данным
            train_mask = group_order <= purged_max_order
            val_mask = np.isin(groups, allowed_val_eras)

            train_indices = np.where(train_mask)[0]
            val_indices = np.where(val_mask)[0]

            yield train_indices, val_indices


cv = PurgedTimeSeriesSplit(n_folds=5, purge_gap=1, embargo=1)

fold_data = []
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, groups=train["era"])):
    
    train_era_range = sorted(train.iloc[train_idx]["era"].unique())
    val_era_range = sorted(train.iloc[val_idx]["era"].unique())
    fold_data.append({
        "fold": fold+1,
        "train_eras": train_era_range,
        "val_eras": val_era_range
    })
    
plt.figure(figsize=(12, len(fold_data) * 1.5))

for item in fold_data:
    fold = item["fold"]
    train_eras = item["train_eras"]
    val_eras = item["val_eras"]

    # Горизонтальные линии
    plt.hlines(fold, min(train_eras), max(train_eras), color='blue', linewidth=10, label='Train' if fold == 1 else "")
    plt.hlines(fold, min(val_eras), max(val_eras), color='red', linewidth=10, label='Val' if fold == 1 else "")

plt.yticks(range(1, len(fold_data) + 1))
plt.xlabel("Era")
plt.ylabel("Fold")
plt.title("Era-wise Cross-Validation Splits")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# В начале study. Пространство: небольшие изменения вокруг baseline
study.set_user_attr("baseline_score", baseline_mean_corr)  # Логирование: baseline_score в user_attr
study.enqueue_trial(standard_large_lgbm_params)
Objective: maximize mean_era_corr

In [ ]:
X = train[feature_set].copy()
y = train['target'].copy()
eras = train['era'].values.copy()

# Инициализируем сплиттер
splitter = PurgedGroupTimeSeriesSplit(n_splits=8, purge=2, embargo=1)

# Собираем результаты разбиений
result = []

unique_eras = sorted(train['era'].unique(), key=lambda x: int(x))
era_to_idx = {era: i for i, era in enumerate(unique_eras)}
eras_idx_arr = np.array([era_to_idx[era] for era in eras])

for train_idx, val_idx in splitter.split(X, y, eras=eras):
    train_eras_idx = sorted(set(era_to_idx[era] for era in train.iloc[train_idx]['era']))
    val_eras_idx = sorted(set(era_to_idx[era] for era in train.iloc[val_idx]['era']))
    
    result.append({
        'train': train_eras_idx,
        'val': val_eras_idx
    })

# Строим график
plt.figure(figsize=(12, 6))

for fold, split in enumerate(result):
    train_eras = split['train']
    val_eras = split['val']
    
    if train_eras:
        plt.hlines(fold, min(train_eras), max(train_eras), color='blue', linewidth=10, label='Train' if fold == 0 else "")
    if val_eras:
        plt.hlines(fold, min(val_eras), max(val_eras), color='red', linewidth=10, label='Val' if fold == 0 else "")

plt.gca().invert_yaxis()
plt.yticks(range(len(result)), [f'Fold {i+1}' for i in range(len(result))])
plt.xlabel("Era Index")
plt.ylabel("Fold")
plt.title("Era-wise Cross-Validation Splits (with Purge & Embargo)")
plt.legend(loc='upper right')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
int(np.ceil(np.log2(64)) + 1)